In [4]:
import re
import sys
from collections import defaultdict
from pathlib import Path


# ──────────────────────────────────────────────────────
# PATTERNS  — matched against each log line
# ──────────────────────────────────────────────────────

# Page attempt:  │   [3/10] "gyms"  page 2  (keys active: 131)
RE_PAGE_ATTEMPT = re.compile(
    r'\[(?P<q_idx>\d+)/\d+\]\s+"(?P<query>[^"]+)"\s+page\s+(?P<page>\d+)'
    r'\s+\(keys active:\s*(?P<keys>\d+)\)'
)

# Page result:   │   └─ Page 2: 20 returned | 5 new in slice | 120 global unique
RE_PAGE_RESULT = re.compile(
    r'└─ Page\s+(?P<page>\d+):\s+(?P<returned>\d+) returned'
    r'\s+\|\s+(?P<new_in_slice>\d+) new in slice'
    r'\s+\|\s+(?P<global_unique>\d+) global unique'
)

# Zero result:   │   └─ 0 results — pagination complete.
RE_ZERO_RESULT = re.compile(r'0 results.*pagination complete')

# Error lines
RE_ERROR_HTTP   = re.compile(r'✗ HTTP (\d+)')
RE_ERROR_MAXRET = re.compile(r'Max retries reached')
RE_ERROR_NOKEYS = re.compile(r'No valid API keys left|Key pool empty')

# Key revoked:   │   ❄ Key ...abc123 revoked (...) — 131 remaining
RE_REVOKE = re.compile(r'❄\s+Key\s+(\S+)\s+revoked\s+\(([^)]+)\)')

# Retry warning: │   ☢ HTTP 429 — ... [2/5] retry in 4s
RE_RETRY = re.compile(r'☢.*?\[(\d+)/\d+\]\s+retry')

# City header:   CITY [3/8]  BOSTON
RE_CITY = re.compile(r'CITY\s+\[\d+/\d+\]\s+(\S+)')

# Zone header:   ┌─ Zone [1/5]  downtown_boston
RE_ZONE = re.compile(r'┌─ Zone\s+\[\d+/\d+\]\s+(\S+)')

# Summary block (may be absent if run was interrupted)
RE_SUMMARY_CALLS   = re.compile(r'Total API calls\s*:\s*([\d,]+)')
RE_SUMMARY_UNIQUE  = re.compile(r'Unique records\s*:\s*([\d,]+)')
RE_SUMMARY_DUPES   = re.compile(r'Duplicate skips\s*:\s*([\d,]+)')
RE_SUMMARY_ERRORS  = re.compile(r'Errors\s*:\s*([\d,]+)')
RE_SUMMARY_KEYS    = re.compile(r'Keys still active\s*:\s*(\d+)\s*/\s*(\d+)')

# Timestamp prefix: "2026-05-22 00:41:14  INFO     "
RE_TIMESTAMP = re.compile(r'^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})')


# ──────────────────────────────────────────────────────
# PARSER
# ──────────────────────────────────────────────────────

def parse_log(log_path: Path, verbose: bool = False) -> None:
    text = log_path.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()

    # ── running state ────────────────────────
    current_city  = "unknown"
    current_zone  = "unknown"
    current_query = "unknown"

    # ── counters ──────────────────────────
    total_attempts   = 0   # log lines with "page N  (keys active:"
    total_successes  = 0   # log lines with "└─ Page N: X returned"
    total_zero       = 0   # "0 results — pagination complete"
    total_errors     = 0   # HTTP error / max-retries / no-keys lines
    total_retries    = 0   # ☢ retry lines
    total_revocations = 0  # ❄ revoke lines

    # per-city attempt counts
    city_attempts: dict[str, int]  = defaultdict(int)
    city_success:  dict[str, int]  = defaultdict(int)
    city_errors:   dict[str, int]  = defaultdict(int)
    city_records:  dict[str, int]  = {}   # last "global unique" seen per city

    # per-query attempt counts (across all cities)
    query_attempts: dict[str, int] = defaultdict(int)

    # error detail log
    error_lines: list[str] = []

    # revoked keys
    revoked_keys: list[tuple[str, str]] = []   # (masked_key, reason)

    # summary block values (from log footer if present)
    summary: dict = {}

    # start/end timestamps
    ts_first = ts_last = None

    # ── parse ──────────────────────────
    for line in lines:
        # timestamps
        ts_m = RE_TIMESTAMP.match(line)
        if ts_m:
            ts = ts_m.group(1)
            if ts_first is None:
                ts_first = ts
            ts_last = ts

        # city / zone tracking
        m = RE_CITY.search(line)
        if m:
            current_city = m.group(1).lower()
            continue

        m = RE_ZONE.search(line)
        if m:
            current_zone = m.group(1)
            continue

        # page attempt
        m = RE_PAGE_ATTEMPT.search(line)
        if m:
            current_query = m.group("query")
            total_attempts += 1
            city_attempts[current_city] += 1
            query_attempts[current_query] += 1
            continue

        # page result (success)
        m = RE_PAGE_RESULT.search(line)
        if m:
            total_successes += 1
            city_success[current_city] += 1
            city_records[current_city] = int(m.group("global_unique"))
            continue

        # zero results
        if RE_ZERO_RESULT.search(line):
            total_zero += 1
            continue

        # errors
        if RE_ERROR_HTTP.search(line) or RE_ERROR_MAXRET.search(line) or RE_ERROR_NOKEYS.search(line):
            total_errors += 1
            city_errors[current_city] += 1
            if verbose:
                error_lines.append(f"  {line.strip()}")
            continue

        # retries (warnings)
        if RE_RETRY.search(line):
            total_retries += 1
            continue

        # key revocations
        m = RE_REVOKE.search(line)
        if m:
            total_revocations += 1
            revoked_keys.append((m.group(1), m.group(2)))
            continue

        # summary block
        for pattern, key in [
            (RE_SUMMARY_CALLS,  "logged_api_calls"),
            (RE_SUMMARY_UNIQUE, "logged_unique"),
            (RE_SUMMARY_DUPES,  "logged_dupes"),
            (RE_SUMMARY_ERRORS, "logged_errors"),
        ]:
            sm = pattern.search(line)
            if sm:
                summary[key] = int(sm.group(1).replace(",", ""))

        sm = RE_SUMMARY_KEYS.search(line)
        if sm:
            summary["keys_remaining"] = int(sm.group(1))
            summary["keys_total"]     = int(sm.group(2))

    # ── failed attempts = attempts that have no matching result line ──
    # (successes + zeros account for all completed page slots)
    completed   = total_successes + total_zero
    failed_slots = max(0, total_attempts - completed)

    # ──────────────────────────────────────────────────────
    # REPORT
    # ──────────────────────────────────────────────────────
    W = 62
    sep  = "═" * W
    thin = "─" * W

    def row(label, value, note=""):
        note_str = f"  ← {note}" if note else ""
        print(f"  {label:<32} {str(value):>10}{note_str}")

    print(sep)
    print(f"  LOG ANALYSIS  {log_path.name}")
    print(sep)
    if ts_first:
        print(f"  Period : {ts_first}  →  {ts_last}")
    print()

    print("  API CALL COUNTS (from log lines)")
    print(thin)
    row("Page attempts logged",     total_attempts,
        "increments before each HTTP call")
    row("  — returned results",     total_successes,
        "got ≥1 place back")
    row("  — returned 0 results",   total_zero,
        "clean end-of-pages")
    row("  — failed / no result",   failed_slots,
        "error or skipped after retry exhaustion")
    print()
    row("Retry warnings fired",     total_retries,
        "backoff retries inside call_api()")
    row("Keys revoked",             total_revocations)
    row("Error lines logged",       total_errors)
    print()

    note_diff = ""
    if "logged_api_calls" in summary:
        diff = total_attempts - summary["logged_api_calls"]
        note_diff = (
            f"matches summary" if diff == 0
            else f"summary says {summary['logged_api_calls']:,}  Δ={diff:+,}"
        )
    row("TOTAL API CALLS (counted)", total_attempts, note_diff)

    if summary:
        print()
        print("  SUMMARY BLOCK (from log footer)")
        print(thin)
        if "logged_api_calls" in summary:
            row("Logged 'Total API calls'",   f"{summary['logged_api_calls']:,}")
        if "logged_unique"    in summary:
            row("Logged unique records",       f"{summary['logged_unique']:,}")
        if "logged_dupes"     in summary:
            row("Logged duplicate skips",      f"{summary['logged_dupes']:,}")
        if "logged_errors"    in summary:
            row("Logged errors",               f"{summary['logged_errors']:,}")
        if "keys_remaining"   in summary:
            row("Keys remaining / total",
                f"{summary['keys_remaining']} / {summary['keys_total']}")

    print()
    print("  BREAKDOWN BY CITY")
    print(thin)
    all_cities = sorted(set(list(city_attempts) + list(city_success)))
    print(f"  {'City':<22} {'Attempts':>9} {'Success':>9} {'Errors':>8}")
    print(f"  {'-'*22} {'-'*9} {'-'*9} {'-'*8}")
    for c in all_cities:
        print(
            f"  {c:<22} {city_attempts[c]:>9,} "
            f"{city_success[c]:>9,} "
            f"{city_errors.get(c, 0):>8,}"
        )

    print()
    print("  BREAKDOWN BY QUERY TYPE")
    print(thin)
    print(f"  {'Query':<30} {'Attempts':>9}")
    print(f"  {'-'*30} {'-'*9}")
    for q, cnt in sorted(query_attempts.items(), key=lambda x: -x[1]):
        print(f"  {q:<30} {cnt:>9,}")

    if total_revocations:
        print()
        print(f"  REVOKED KEYS ({total_revocations})")
        print(thin)
        seen = {}
        for masked, reason in revoked_keys:
            seen[masked] = reason
        for masked, reason in seen.items():
            print(f"  {masked}  —  {reason}")

    if verbose and error_lines:
        print()
        print(f"  ERROR LINES ({len(error_lines)})")
        print(thin)
        for el in error_lines[:50]:
            print(el)
        if len(error_lines) > 50:
            print(f"  ... and {len(error_lines) - 50} more (run without --verbose cap)")

    print()
    print(sep)
    print(
        f"  ANSWER: {total_attempts:,} API calls were fired"
        + (f"  ({failed_slots} failed, {total_successes} returned data)" if failed_slots else "")
    )
    print(sep)


# ──────────────────────────────────────────────────────
# ENTRY POINT
# ──────────────────────────────────────────────────────

def main():
    args = sys.argv[1:]
    if not args or args[0] in ("-h", "--help"):
        print(__doc__)
        sys.exit(0)

    verbose   = "--verbose" in args
    # log_files = [a for a in args if not a.startswith("--") dynamically getting log files]
    log_files = ["/content/20260528_004701_scraper.log", "/content/scraper (1).log", "/content/scraper-detroit.log", "/content/scraper.log"]

    if not log_files:
        print("ERROR: provide a path to scraper.log")
        sys.exit(1)

    for path_str in log_files:
        p = Path(path_str)
        if not p.exists():
            print(f"ERROR: file not found — {p}")
            sys.exit(1)
        parse_log(p, verbose=verbose)


if __name__ == "__main__":
    main()

══════════════════════════════════════════════════════════════
  LOG ANALYSIS  20260528_004701_scraper.log
══════════════════════════════════════════════════════════════
  Period : 2026-05-28 00:47:01  →  2026-05-28 02:21:29

  API CALL COUNTS (from log lines)
──────────────────────────────────────────────────────────────
  Page attempts logged                   3713  ← increments before each HTTP call
    — returned results                    255  ← got ≥1 place back
    — returned 0 results                    1  ← clean end-of-pages
    — failed / no result                 3457  ← error or skipped after retry exhaustion

  Retry warnings fired                      0  ← backoff retries inside call_api()
  Keys revoked                              0
  Error lines logged                        0

  TOTAL API CALLS (counted)              3713

  BREAKDOWN BY CITY
──────────────────────────────────────────────────────────────
  City                    Attempts   Success   Errors
  -------